<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_2_MLR_RegressionTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames, Iowa: Tree-Based Methods
## Decision Trees, Random Forests, and Gradient Boosting

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

---

If you have not seen this material before, I highly recommend the StatQuest video on regression trees. It is short and very good at showing how trees are actually built: https://www.youtube.com/watch?v=g9c66TUylZ4

Pay particularly close attention to the description of how residuals are used to decide where splits go (around the 7-minute mark).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Trees do not need one-hot encoding, scaling, or log-transformed features, so
# instead of importing the full ames_cleaning.py pipeline (built for linear
# models), we apply only Part 1's structural cleaning steps here.
url = 'https://raw.githubusercontent.com/bsheese/cs377/main/data/data_housing_ames.txt'
df = pd.read_csv(url, sep='\t')

# Drop unusual sales flagged by the dataset author, identifiers, and the same
# sparse/constant columns dropped in Part 1
df = df[df['Gr Liv Area'] < 4000]
df = df.drop(['Order', 'PID'], axis=1)
sparse_drops = ['Pool QC', 'Pool Area', 'Misc Feature', 'Misc Val', 'Alley', 'Fence']
constant_drops = ['Street', 'Utilities', 'Condition 2', 'Roof Matl', 'Heating', 'Low Qual Fin SF', '3Ssn Porch']
df = df.drop(sparse_drops + constant_drops + ['Garage Yr Blt'], axis=1, errors='ignore')

# MS SubClass is a numeric code, not a magnitude
df['MS SubClass'] = df['MS SubClass'].astype(str)

# Meaningful NAs: for these columns, missing means "feature not present"
for col in ['Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2',
            'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond', 'Fireplace Qu']:
    df[col] = df[col].fillna('None')

# Convert the remaining text columns to pandas 'category' dtype
obj_cols = df.select_dtypes(include=['object', 'string']).columns
df[obj_cols] = df[obj_cols].astype('category')

print(f"DataFrame shape: {df.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# For Decision Trees and Random Forests, we need numeric data.
# Unlike linear models, trees do not require one-hot encoding. They can work
# with categorical features as long as the categories are represented as numbers.
# We use .cat.codes to convert each category to an integer (e.g., 'PConc' → 0, 'CBlock' → 1).
# This preserves the categorical nature without exploding the feature count.
# Note: Later models (HistGradientBoosting and XGBoost) will accept category dtypes natively.
X = df.drop(columns=['SalePrice']).copy()
y = df['SalePrice'].map(np.log)

# Convert category columns to numeric codes for models that do not support them natively
for col in X.select_dtypes(include=['category']).columns:
    X[col] = X[col].cat.codes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Setup complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## Why Trees?

Across Parts 1 through 5 we built and refined linear regression models: selecting features, handling multicollinearity, applying regularization, tuning hyperparameters, and finally evaluating it all honestly with nested CV. But linear models carry one constraint we never escaped. They assume the relationship between each feature and the target is fundamentally a straight line. When reality curved, as it did back in 17_1_5, our only option was to transform variables by hand until the relationship straightened out.

Tree-based models take a completely different approach. Instead of fitting a single equation with coefficients, a tree splits the data into regions and predicts the average target value within each region. That means the model can capture on its own the kinds of patterns that took manual effort in linear regression. For example:

*   **Diminishing returns.** Increasing `Garage Area` from 0 to 400 sq ft adds a lot of value. Increasing it from 1000 to 1400 adds much less. A tree captures this by splitting at different thresholds.
*   **Threshold effects.** Houses built after a certain year might command a specific premium because of building codes or styles. A tree finds these breakpoints automatically.
*   **Feature interactions.** `Overall Qual` might matter more in some neighborhoods than others. A tree captures this by splitting on one feature, then another, within each branch. (The 17_3 notebook shows what it takes to capture exactly this kind of effect in a linear model: you have to *hand-build* an interaction term. Trees find them on their own.)

In this notebook we are going to explore three tree-based methods, each building on the last:

1.  **Decision trees:** a single tree. Highly interpretable, but prone to overfitting.
2.  **Random forests:** hundreds of trees trained in parallel and averaged together. Stable and robust.
3.  **Gradient boosting:** trees built one after another, each one correcting the errors of the last. Often the most accurate.

We will focus on regression trees (predicting a continuous value). Classification trees, which predict categories, are covered in the next module.

## The 20 Questions Analogy

A decision tree is a structured way of asking a series of yes/no questions to arrive at a prediction. It is exactly like the game "20 Questions," where one person thinks of an object and the other asks yes/no questions to figure out what it is.

### Guessing a Number

Imagine you are trying to guess a secret number between 1 and 100. The best strategy is to split the remaining possibilities in half at each step:

1. **First question:** "Is the number > 50?"
2. **If yes:** ask "Is it > 75?"
3. **If no:** ask "Is it > 25?"

This kind of binary split is the foundation of decision trees.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def draw_decision_tree():
    """Draw a simple decision tree for the number guessing game."""
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title("Decision Tree: Guessing a Number Between 1-100", fontsize=16, fontweight='bold', pad=20)

    def draw_node(x, y, text, color='lightblue'):
        """Draw a rectangular node."""
        rect = mpatches.FancyBboxPatch((x-0.6, y-0.4), 1.2, 0.8,
                                        boxstyle="round,pad=0.05",
                                        facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(x, y, text, ha='center', va='center', fontsize=11, fontweight='bold')

    def draw_edge(x1, y1, x2, y2, label):
        """Draw an edge between nodes with a label."""
        ax.annotate('', xy=(x2, y2+0.4), xytext=(x1, y1-0.4),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2
        ax.text(mid_x, mid_y, label, fontsize=9, ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='none', alpha=0.8))

    level_1_y = 8
    level_2_y = 6
    level_3_y = 4
    level_4_y = 2

    draw_node(7, level_1_y, "Is > 50?\n(50)", 'lightblue')

    draw_node(3.5, level_2_y, "Is > 25?\n(25)", 'lightgreen')
    draw_node(10.5, level_2_y, "Is > 75?\n(75)", 'lightgreen')

    draw_edge(7, level_1_y-0.4, 3.5, level_2_y+0.4, 'Yes')
    draw_edge(7, level_1_y-0.4, 10.5, level_2_y+0.4, 'No')

    draw_node(1.75, level_3_y, "Is > 12?\n(12)", 'lightyellow')
    draw_node(5.25, level_3_y, "Is > 37?\n(37)", 'lightyellow')
    draw_node(8.25, level_3_y, "Is > 62?\n(62)", 'lightyellow')
    draw_node(12.75, level_3_y, "Is > 87?\n(87)", 'lightyellow')

    draw_edge(3.5, level_2_y-0.4, 1.75, level_3_y+0.4, 'Yes')
    draw_edge(3.5, level_2_y-0.4, 5.25, level_3_y+0.4, 'No')
    draw_edge(10.5, level_2_y-0.4, 8.25, level_3_y+0.4, 'Yes')
    draw_edge(10.5, level_2_y-0.4, 12.75, level_3_y+0.4, 'No')

    draw_node(0.875, level_4_y, "Leaf\n(1-12)", 'lightcoral')
    draw_node(2.625, level_4_y, "Leaf\n(13-25)", 'lightcoral')
    draw_node(4.375, level_4_y, "Leaf\n(26-37)", 'lightcoral')
    draw_node(6.125, level_4_y, "Leaf\n(38-50)", 'lightcoral')
    draw_node(7.875, level_4_y, "Leaf\n(51-62)", 'lightcoral')
    draw_node(9.625, level_4_y, "Leaf\n(63-75)", 'lightcoral')
    draw_node(11.375, level_4_y, "Leaf\n(76-87)", 'lightcoral')
    draw_node(13.125, level_4_y, "Leaf\n(88-100)", 'lightcoral')

    for x in [1.75, 5.25, 8.25, 12.75]:
        draw_edge(x, level_3_y-0.4, x-0.875, level_4_y+0.4, 'Yes')
        draw_edge(x, level_3_y-0.4, x+0.875, level_4_y+0.4, 'No')

    ax.text(7, 0.3, "Each path from root to leaf represents a sequence of yes/no answers",
            ha='center', fontsize=10, style='italic', color='gray')

    plt.tight_layout()
    plt.show()

draw_decision_tree()

## Key Decision Tree Terminology

| Term | Definition | In Our Game |
|-----|-------------|--------------|
| **Root** | The starting node | "Is > 50?" |
| **Split** | A decision point | Each yes/no question |
| **Branch** | A path from one node to another | The "Yes" or "No" answer |
| **Leaf** | A terminal node with no children | The final guess range |
| **Depth** | Number of levels in the tree | How many questions asked |
| **Impurity** | How mixed the groups are at a node | How close the range is to one answer |

## How This Applies to Real Data

In the number-guessing game we split exactly in half at each step, because every number was equally likely. Real data is not so symmetric, so the algorithm has to *choose* its questions. At each node it hunts for the feature and threshold whose split most improves the situation. For classification that means best separating the classes. For regression, our case, it means most reducing the spread (the variance) of the target within each resulting group. The best split is found automatically, by trying the possibilities and keeping the best one.

So the whole algorithm is the game, played greedily: start at the root with the single most informative question, let each answer lead to the next question, and stop at a leaf, where the prediction lives.

And the game has the same failure mode our linear models did, in a different costume. A tree that asks too *few* questions underfits: it never learns enough about a house to price it well. A tree that asks too *many* memorizes the training set: every house ends up in its own private leaf. Depth is the tree's complexity dial, and finding its sweet spot is the same bias-variance balancing act we did for alpha in Part 4 and for polynomial degree in 17_1_6. We are about to watch that play out on real data.

## How a Regression Tree Works with the Ames Data

The model tries to predict a house's price by asking a series of yes/no questions:

1.  "Is the house larger than 2,000 sq ft?"
2.  "Is the Overall Quality greater than 7?"
3.  "Does it have a finished basement?"

Each question splits the data into two groups. The model keeps splitting until it reaches a stopping point. At that point, it predicts the **average price** of the houses in that final group.

Here is how the pieces map to the terminology from the table above:

*   **Node (decision node):** a point where the tree asks a question. The root node is the very first split.
*   **Branch:** the "yes" or "no" path that leads from one node to the next.
*   **Leaf (terminal node):** an endpoint where the tree stops splitting. The prediction is the average target value of all the training houses that landed in that leaf.

You can picture each house flowing down through a branching path of decision nodes until it settles into a leaf, where it receives its predicted price.

### How Does the Tree Choose Its Questions?

The tree does not ask questions at random. At each node, it tries every possible split point on every feature and picks the one that reduces prediction error the most (technically, the split that minimizes the variance of the target values in the two resulting groups). Then it repeats that process inside each group, over and over, until it is done.

### The Overfitting Problem

If we let the tree keep splitting without any limits, it will eventually create a separate leaf for nearly every individual house in the training set. The result is a model that achieves a perfect $R^2$ of 1.0 on the training data by memorizing it, and performs poorly on any new data. This is classic overfitting. The model has learned the noise along with the pattern.

The key hyperparameter for controlling this is `max_depth`, which limits how many questions the tree is allowed to ask along any path. Let's see what happens as we vary it.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# 1. Define a list of max_depth values to experiment with
# We test a range from shallow (underfitting) to unconstrained (overfitting)
# to find the depth that best balances bias and variance.
max_depth_values = [3, 5, 7, 9, 11, None]

# 2. Initialize empty lists for results
depths = []
train_r2s = []
test_r2s = []

print("--- Decision Tree Regressor Performance by Max Depth ---")

# 3. Loop through each max_depth value
for depth in max_depth_values:
    # 4a. Instantiate a DecisionTreeRegressor
    dt_regressor = DecisionTreeRegressor(max_depth=depth, random_state=42)

    # 4b. Fit the regressor to the training data
    dt_regressor.fit(X_train, y_train)

    # 4c. Calculate the R² score on the training data
    train_r2 = dt_regressor.score(X_train, y_train)
    train_r2s.append(train_r2)

    # 4d. Calculate the R² score on the testing data
    test_r2 = dt_regressor.score(X_test, y_test)
    test_r2s.append(test_r2)

    # 4e. Append the current max_depth value
    depths.append(str(depth) if depth is not None else 'None (Unconstrained)')

    print(f"Max Depth: {str(depth) if depth is not None else 'None (Unconstrained)':<20} | Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")

# 5. Create a Pandas DataFrame from the collected results
dt_results_df = pd.DataFrame({
    'Max Depth': depths,
    'Train R2': train_r2s,
    'Test R2': test_r2s
})

# print("\nDecision Tree Results DataFrame:")
# print(dt_results_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Prepare data for plotting
plot_df = dt_results_df.copy()
# Convert 'Max Depth' column to numeric for plotting, treating 'None (Unconstrained)' as a large number or specific category
# For visualization, it's often better to represent 'None' distinctly or order it last.
# For a numeric plot, we can assign an arbitrary large number or handle it as a categorical value.
# Let's convert it to a numeric type, assigning a distinct value for 'None' for proper ordering in the plot
plot_df['Max Depth Num'] = plot_df['Max Depth'].replace({'None (Unconstrained)': 15}).astype(float) # Assigning 15 as an example, larger than 11

plt.figure(figsize=(10, 6))
sns.lineplot(data=plot_df, x='Max Depth Num', y='Train R2', marker='o', label='Train R²')
sns.lineplot(data=plot_df, x='Max Depth Num', y='Test R2', marker='o', label='Test R²')

# Customize x-axis ticks to show original labels
x_tick_labels = plot_df['Max Depth'].tolist()
x_tick_positions = plot_df['Max Depth Num'].tolist()
plt.xticks(ticks=x_tick_positions, labels=x_tick_labels)

plt.title('Decision Tree R² Score vs. Max Depth')
plt.xlabel('Max Depth')
plt.ylabel('R² Score')
plt.grid(True)
plt.legend()
plt.show()

## Interpreting the Results

The table and plot above show the same bias-variance tradeoff we saw in Part 4, now expressed through tree depth:

*   **Depth 3 (high bias / underfitting):** both training and test scores are moderate (about 0.71 train / 0.75 test). The tree is too shallow to capture the complexity of the data. It is asking too few questions.
*   **Depth 5 (sweet spot):** the test $R^2$ peaks at depth 5 (0.822). The tree is complex enough to capture meaningful patterns but not so complex that it memorizes noise. This is the best generalization we see.
*   **Depth 7 to 11 (increasing overfitting):** the training score keeps climbing (0.91 to 0.98), but the test score drifts downward. The gap between the two lines widens. The tree is now fitting noise specific to the training set.
*   **Depth None (unconstrained / memorization):** the training $R^2$ hits 0.9999. The tree has essentially memorized every training house. The test $R^2$ falls to 0.794, giving back everything the extra depth was supposed to buy. This is the most extreme form of overfitting available to a single tree.

The plot makes the divergence clear: the training score climbs steadily toward 1.0, while the test score peaks at depth 5 and then declines. The gap between the two lines is the overfitting penalty.

This is the same pattern we saw with polynomial features in 17_1_6. As model complexity increases, training performance always improves, but test performance eventually gets worse. The difference is that with trees, complexity is controlled by a single intuitive parameter (depth) rather than by hand-building polynomial terms.

So how do we keep a single tree from overfitting? One approach is to limit its depth. A more powerful approach is to build many trees and combine their predictions. That is the idea behind random forests.

## Visualizing a Single Tree
Before we get to forests, take a look at the nodes in this single tree of depth 5. In the first few levels you can see the specific values the tree chose for each decision.

In [ ]:
# visualize our depth 5 tree

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

# Instantiate a DecisionTreeRegressor with max_depth=5 and random_state=42
dt_depth5 = DecisionTreeRegressor(max_depth=5, random_state=42)

# Fit the model using X_train and y_train
dt_depth5.fit(X_train, y_train)

# Use plt.figure to set a large figure size to ensure the tree is legible
plt.figure(figsize=(20, 10))

# Call plot_tree passing the fitted model
plot_tree(dt_depth5,
          feature_names=X_train.columns,
          filled=True,
          rounded=True,
          fontsize=10)

# Display the plot
plt.title('Decision Tree Visualization (Max Depth = 5)')
plt.show()

### Reading the Tree

Take a moment to look at the tree visualization above. Here is what to look for:

*   **The root node (top box):** this is the tree's first and most important split. Which feature did it choose? According to the tree, this is the single strongest predictor of house price.
*   **Feature repetition:** notice that the same features appear multiple times at different depths. A feature might be used to split at depth 1, and then appear again at depth 3 within a specific branch. This is how trees capture interactions: the meaning of a feature can change depending on the path you took to reach it. This is the same thing 17_3 models explicitly with its `bmi * smoker` interaction term, except the tree finds it without being told.
*   **The leaves (bottom boxes):** each leaf shows a `value` (the predicted log-price for houses in that group) and `samples` (how many training houses landed there). Leaves with very few samples are a warning sign. The model is making predictions based on a tiny number of examples.

Even at just depth 5, this tree is already getting hard to read. At depths of 10 or more, a visualization would be completely illegible. This is one reason why single trees, while interpretable at shallow depths, become "black boxes" as they grow deeper. It is also a preview of why we need ensemble methods: if one tree is hard to read at depth 5, imagine trying to interpret a forest of 200 trees.

## Random Forests: Ensemble Learning via Bagging

A single decision tree has a fundamental weakness: it overfits. Left unchecked, it memorizes the training data. Even when we limit its depth, a single tree is highly sensitive to the specific houses it was trained on. Change a few data points and the entire tree structure can shift. In the language of 17_1_6, a single tree has high variance.

So what if, instead of building one carefully pruned tree, we built many imperfect trees and averaged their predictions?

Think of it like the wisdom of the crowd. Ask one person to guess the price of a house and they might be way off. Ask 100 people and average their guesses, and the average is often surprisingly accurate. The individual errors cancel out. That is the core idea behind a random forest.

But there is a catch. If every tree sees the same data and considers the same features, they will all make the *same* mistakes, and averaging identical predictions doesn't help. To make the crowd genuinely diverse, the random forest introduces two sources of randomness.

**1. Bootstrap aggregating (bagging): randomizing the data**

Each tree is trained on a random sample of the training data, drawn *with replacement*. This is exactly the bootstrap resampling we used in 17_1_2 to measure how much a slope wiggles. Here it is used to deliberately grow a different tree each time. Some houses appear multiple times in a tree's training set, and others don't appear at all. Tree #47 might never see the most expensive mansion in the dataset, so it develops a different view of what drives prices. Tree #83 might get an unusual concentration of older homes, making it better at valuing them.

**2. Feature randomness: randomizing the features**

Even with different data, a dominant feature like `Overall Qual` would still be chosen as the first split by nearly every tree. To prevent this, at every node each tree is only allowed to consider a random subset of the available features. If `Overall Qual` isn't in the subset, the tree is forced to find the next-best split. This decorrelates the trees and makes the forest explore many different patterns in the data.

Because every tree is built independently (different data, different feature subsets), the trees don't need to communicate or learn in sequence. They can be trained in parallel across multiple CPU cores.

From a machine learning perspective, the random forest's main job is variance reduction. Individual trees overfit and give noisy predictions. When you average hundreds of diverse, mostly uncorrelated opinions, the noise cancels out. This is the same thing we saw in the bias-variance simulation in 17_1_6, where averaging many wiggly fits produced a much more stable curve. The result is a model that is far more stable and accurate than any single tree could be.

### Tuning the Random Forest

We are going to use GridSearchCV to find the best combination of three key hyperparameters:

*   `n_estimators`: how many trees to build. More trees means more stable, but slower.
*   `max_depth`: how deep each tree can grow. Unlike a single tree, random forests can often benefit from deeper (or even unlimited depth) trees, because the averaging controls the overfitting.
*   `max_features`: how many features each tree considers at each split. `'sqrt'` means the square root of the total number of features, which is the default and most common choice.

Note: unlike linear models, tree-based models do **not** require feature scaling. A split at "Garage Area > 500" works the same whether the column is measured in square feet or z-scores. We pass the data directly to the model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'max_features': [1.0, 'sqrt']
}

print("--- Tuning Random Forest ---")

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train)

print(f"\nBest Parameters: {rf_grid.best_params_}")
print(f"Best CV R²:      {rf_grid.best_score_:.4f}")

best_rf = rf_grid.best_estimator_

### Interpreting the Random Forest Results

Our tuned random forest achieved a CV $R^2$ of **0.8797**, an improvement over the single decision tree's best test $R^2$ of **0.8220** at depth 5. This is the payoff of ensembling. By averaging many diverse trees, we reduced the variance that limited any single tree.

Notice the best parameters:

*   **`max_depth=None`**: the forest chose unlimited tree depth. For a single tree, this was the worst setting (test $R^2$ = 0.794). But in a forest, the averaging controls the overfitting, so deeper trees can capture more nuance without harming generalization.
*   **`max_features='sqrt'`**: each tree considers only about 8 features (√65 ≈ 8) at each split. This forces diversity. If it had chosen `1.0` (all features), the trees would be more correlated and the ensemble less effective.
*   **`n_estimators=200`**: the larger forest won, though typically by a whisker over 100 trees. Averaging more trees rarely hurts; it just costs more compute.

Note: here the grid chose `max_features='sqrt'` (about 8 of 65 features per split), the common default. For regression, some practitioners prefer roughly 1/3 of the features. It is worth experimenting with this setting.

## Gradient Boosting: Ensemble Learning via Sequential Refinement

Random forests build trees in parallel and hope the crowd is wise. There is another approach: what if each tree could learn from the mistakes of the trees that came before it?

That is the idea behind gradient boosting. Instead of independent trees voting in parallel, boosting builds trees one after another, where each new tree is trained specifically to correct the errors of the ensemble so far.

### Example of Gradient Boosting

1.  **Initial prediction.** The model starts with the simplest possible guess, the average log-price of all training houses. Let's say that is about 12.0. Every house gets this same prediction.
2.  **Calculate residuals.** The model looks at its errors. House #42 (a small fixer-upper) was predicted at 12.0 but actually sold for 11.2. Its residual is −0.8. House #101 (a renovated colonial) was predicted at 12.0 but sold for 12.9. Its residual is +0.9.
3.  **Train tree #2 on the residuals.** The next tree is not trained to predict house prices. It is trained to predict the residuals. It learns patterns like "small houses in this neighborhood tend to be overpriced by the average" and "renovated homes in this area are underpriced."
4.  **Update the predictions.** Tree #2's predictions are added to the running total, nudging each house closer to its true price. House #42's new prediction: 12.0 + (−0.7) = 11.3. Closer.
5.  **Repeat.** The model recalculates the new, smaller residuals and trains tree #3 on those. Each tree makes progressively smaller corrections. After 200 trees, the residuals are tiny.

You can think of this like a specialized assembly line. The first worker cuts out the rough shape. Every worker after that focuses only on sanding down the exact rough edges left by the person before them.

### Bias vs. Variance

This is the key difference between the two ensemble approaches:

| | Random Forest | Gradient Boosting |
|---|---|---|
| **Strategy** | Build many deep trees in parallel, then average | Build many shallow trees sequentially, each correcting errors |
| **Target** | Reduces **variance** (averages out noise) | Reduces **bias** (chips away at underfitting) |
| **Analogy** | Wisdom of the crowd | Specialized assembly line |

---

*Note on implementation:* we are going to use `HistGradientBoostingRegressor`, scikit-learn's modern boosting implementation. It speeds up the algorithm by grouping continuous data into discrete bins (histograms) before making splits, which makes it fast even on large datasets.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# HistGradientBoostingRegressor handles categorical features natively
# so we can use our original X (with category dtypes) directly.
# We re-split to get clean train/test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

hgb_model = HistGradientBoostingRegressor(
    max_iter=200,        # Number of sequential trees to build
    learning_rate=0.1,   # How much each tree contributes (smaller = more trees needed, but more stable)
    random_state=42
)

hgb_model.fit(X_train, y_train)

y_pred = hgb_model.predict(X_test)

# Note: RMSE is in log-price units since our target is log-transformed.
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"HistGradientBoosting Performance:")
print(f"--- Root Mean Squared Error (RMSE): {rmse:,.2f}")
print(f"--- R-squared (R²): {r2:.4f}")

### How Robust Are Tree Models to Messy Data?

In Parts 1 and 2 we spent two full notebooks cleaning the Ames dataset: dropping uninformative features, handling missing values, encoding categories, and removing outliers. For linear regression, that cleaning was essential.

Tree-based models are more forgiving. They don't require one-hot encoding, they handle non-linear relationships automatically, and some implementations (like `HistGradientBoostingRegressor`) can even handle missing values natively.

Let's test that claim. Below, we load the raw Ames data, with only the extreme outlier removal, and run HGB on it with almost no preprocessing. The only steps are: log-transform the target and convert object columns to category dtype.

In [ ]:
# Data source
url = 'https://raw.githubusercontent.com/bsheese/cs377/main/data/data_housing_ames.txt'

# Loading the dataframe
df_raw = pd.read_csv(url, sep='\t')

# Initial cleaning: remove extreme outliers per the author's recommendation
df = df_raw.loc[df_raw['Gr Liv Area'] < 4000, :].copy()

# Define features and target
X = df.drop(columns=['SalePrice'])
y = df['SalePrice'].map(np.log)

# Convert object columns to 'category' dtype for native support
obj_cols = X.select_dtypes(include=['object', 'string']).columns
X[obj_cols] = X[obj_cols].astype('category')

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train with categorical support
hgb_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.1,
    random_state=42,
    categorical_features='from_dtype'
)

hgb_model.fit(X_train, y_train)

y_pred = hgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"HistGradientBoosting Performance (Raw Data with Categories):")
print(f"Root Mean Squared Error (RMSE): {rmse:,.4f}")
print(f"R-squared (R²): {r2:.4f}")

The raw-data model achieved an $R^2$ of **0.9449**, without any cleaning. This demonstrates a key advantage of tree-based methods: they are remarkably robust to messy, unprocessed data. You still benefit from thoughtful feature engineering, but you don't need it to get strong results.

## XGBoost: eXtreme Gradient Boosting

If standard gradient boosting is a specialized assembly line, XGBoost is the same assembly line with better tools, tighter quality control, and a faster conveyor belt.

XGBoost (eXtreme Gradient Boosting) does not change the core idea of boosting. Trees are still built one by one to correct the residuals of the previous trees. But it improves the framework in three important ways.

**1. Built-in regularization**

Remember Ridge and Lasso from Part 3? They penalized large coefficients to prevent overfitting. XGBoost applies the same idea to trees. When evaluating a potential split, it doesn't just ask "does this reduce error?" It also asks "does this make the tree too complex?" It applies L1 (`reg_alpha`) and L2 (`reg_lambda`) penalties on the leaf weights, which shrinks extreme predictions and prunes weak splits. This means XGBoost can safely use deeper trees without overfitting as badly as standard boosting.

**2. Speed**

Boosting is inherently sequential. Tree 2 can't start until tree 1 finishes. That makes it hard to parallelize. XGBoost gets around this by parallelizing the *internal* work of building each tree. Finding the best split requires sorting thousands of values, which is expensive. XGBoost uses an approximate algorithm and parallelized feature sorting to evaluate splits across multiple CPU threads at once. As a result it trains much faster than standard boosting.

**3. Native missing data handling**

Like `HistGradientBoostingRegressor`, XGBoost handles missing values without imputation. At each split, it tests sending missing values left or right and learns which direction reduces error more. If a house is missing its `Garage Quality` score, XGBoost figures out whether that absence itself is informative.

In [ ]:
from xgboost import XGBRegressor

# Use the same cleaned raw data from the HGB section above
X = df.drop(columns=['SalePrice'])
y = df['SalePrice'].map(np.log)

# Convert object columns to 'category' dtype
obj_cols = X.select_dtypes(include=['object', 'string']).columns
X[obj_cols] = X[obj_cols].astype('category')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    reg_alpha=0.1,     # L1 regularization (like Lasso)
    reg_lambda=1.0,    # L2 regularization (like Ridge)
    enable_categorical=True,  # Native pandas 'category' dtype support
    tree_method='hist',       # Histogram binning for speed (same idea as HGB)
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"XGBoost Performance (with Native Categorical Support):")
print(f"Root Mean Squared Error (RMSE): {rmse:,.2f}")
print(f"R-squared (R²): {r2:.4f}")

### Comparing the Three Boosting Approaches

How did XGBoost ($R^2$ = 0.9418) compare to `HistGradientBoostingRegressor` ($R^2$ = 0.9449 on the same raw data)? They are very close, close enough that the ranking can flip from run to run, and here HGB actually edges XGBoost out. On a dataset this size, both are excellent choices. XGBoost tends to shine on larger datasets and in competitive settings (like Kaggle) where every fractional improvement matters and its regularization gives it an edge.

Both boosting models significantly outperformed our random forest ($R^2$ = 0.8797) and our best single decision tree ($R^2$ = 0.8220). For this kind of structured tabular data, the sequential, error-correcting approach of boosting is simply more powerful than parallel averaging.

## Feature Importance

Because complex ensembles like random forests and XGBoost build hundreds of deeply layered trees, they are often criticized as "black boxes." Unlike a linear model, you cannot look at an equation to understand how the final prediction was made. But these algorithms do keep internal records of their decision-making, which we can extract as **feature importance**.

---

To understand how this works, it helps to contrast it with what we had in the linear models.

**Linear model (coefficients)**

In a linear regression, you get a coefficient. It is a direct multiplier. A coefficient of 50 on "Square Footage" means exactly: for every 1 additional square foot, the house price increases by \$50. It is highly interpretable, but it forces a rigid, linear assumption onto the data.

**Tree model (impurity / error reduction)**

Tree ensembles do not calculate linear multipliers. They calculate usefulness. Every time a tree creates a decision node (say, "Is Overall Quality > 7?"), it measures exactly how much that split reduced the model's overall prediction error (often called decreasing "impurity" or "information gain").

If a split on "Roof Material" barely improves the model's accuracy, it gets a very low score. If a split on "Neighborhood" suddenly makes the subsequent predictions much more accurate, it gets a large score for that split.

Across an ensemble of 200 trees, a feature like "Square Footage" might be used to make thousands of different splits. The algorithm adds up the total error reduction attributed to "Square Footage" across every tree, and divides it by the total error reduction of all features combined.

---
**Relative predictive contribution**

Feature importance is usually expressed as a percentage or a relative score (summing to 1.0). If "Overall Quality" has a score of 0.40, that single feature was responsible for 40% of the model's total error reduction.

There is a fundamental limitation to keep in mind. Feature importance tells you what the model relied on, but it does not tell you *how* the feature affects the price. It will tell you that "Lot Size" is a big driver of the model's predictions, but unlike a linear coefficient, the importance score alone won't tell you whether a larger lot increases or decreases the price. It only tells you that the algorithm leaned heavily on that information to reach its answer.

In [ ]:
# 1. Extract the importance scores
# XGBoost calculates these automatically during training based on how much
# each feature contributed to reducing the model's overall error.
importances = xgb_model.feature_importances_

# 2. Map the scores to their corresponding feature names
# We use X_train.columns to ensure the names line up exactly with the data the model saw.
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
})

# 3. Sort the features from most important to least important
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# 4. Isolate the Top 15 features (to keep the chart readable)
top_15_features = importance_df.head(15)

# 5. Visualize the results using Seaborn and Matplotlib
plt.figure(figsize=(10, 8))
sns.barplot(
    x='Importance',
    y='Feature',
    data=top_15_features,
)

# Formatting the plot for readability
plt.title('Top 15 Feature Importances (XGBoost)', fontsize=16, weight='bold')
plt.xlabel('Relative Importance (Contribution to Error Reduction)', fontsize=12)
plt.ylabel('Housing Feature', fontsize=12)
plt.xlim(0, top_15_features['Importance'].max() * 1.1) # Add a little padding to the right
plt.tight_layout()
plt.show()

# Print the exact scores for the top 5 features
print("Top 5 Drivers of House Price (According to XGBoost):")
for index, row in top_15_features.head(5).iterrows():
    # Formatting as a percentage to make the relative importance clearer
    print(f"- {row['Feature']}: {row['Importance'] * 100:.2f}%")

### Interpreting the Results

*   **`Overall Qual` (45.76%):** this single feature accounts for nearly half of the model's learning. It dominated our linear models too. It is simply the strongest predictor of price in this dataset. But notice the difference: in linear regression we could say exactly how much each quality point was worth. Here we only know it was the most important.
*   **`Garage Cars` (12.07%)**
*   **`Central Air` (7.86%)**
*   **`Kitchen Qual` (6.33%)**
*   **`Gr Liv Area` (2.98%):** living area is important, but its importance is relatively low here. Why? Because `Overall Qual` and `Garage Cars` are correlated with size. Bigger houses tend to have higher quality ratings and larger garages. The model is using those correlated features instead, which is a symptom of the multicollinearity issue we discuss next.

### Important Caveat: Correlated Features

Feature importance in tree models can be misleading when features are correlated.

If two features (say `Gr Liv Area` and `Total Bsmt SF`) are highly correlated, the model only needs one of them to make the primary split. That feature gets all the importance credit, while the correlated feature gets none, even though both are equally useful.

We saw this exact issue in Part 2. The VIF table flagged `Total_Square_Footage` and `Log_Gr Liv Area` as substantially overlapping (the two highest VIFs in the table), and we ended up dropping one. In our XGBoost feature importance plot, `Gr Liv Area` appears at 2.98% while `Total Bsmt SF` doesn't appear in the top 15 at all. This doesn't mean basement size is unimportant. It likely means XGBoost chose `Gr Liv Area` for its splits and `Total Bsmt SF` got no credit.

**What to do:**
- Compare feature importance across multiple models (random forest vs. XGBoost, for example). If different models pick different correlated features, that is a warning sign.
- Accept that importance scores are relative, not absolute. They tell you which features the model relied on, not which features are inherently most valuable.

## XGBoost Hyperparameter Tuning: Nested Cross-Validation

Just as we did with the linear models in Part 5, we are going to use nested cross-validation to get an unbiased estimate of XGBoost's performance. This makes sure our score isn't inflated by the hyperparameter search.

XGBoost learns its internal rules directly from the data, but it cannot choose its own external settings (the hyperparameters). Settings like `n_estimators`, `learning_rate`, and `max_depth` have to be tuned.

**1. Inner loop (tuning):** within each training fold, a grid search tests every combination of hyperparameters to find the best settings for that specific subset of data.

**2. Outer loop (evaluation):** the model, using the best parameters found by the inner loop, is evaluated on a separate holdout fold. This makes sure the performance score isn't just "lucky" because of the hyperparameter tuning.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold

# 1. Define the Hyperparameter Grid for the INNER loop
param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'n_estimators': [100, 300]
}

# 2. Initialize the Base Model
base_xgb = XGBRegressor(
    reg_alpha=0.1,
    reg_lambda=1.0,
    enable_categorical=True,
    tree_method='hist',
    random_state=42
)

# 3. Setup Nested Cross-Validation
# Outer loop: evaluates performance
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Inner loop: tunes hyperparameters
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

inner_search = GridSearchCV(
    estimator=base_xgb,
    param_grid=param_grid,
    cv=inner_cv,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)

# 4. Execute Nested CV (Outer Loop)
print("Starting Nested Cross-Validation... this will take a moment.")
nested_scores = cross_val_score(
    inner_search, X, y, cv=outer_cv, scoring='r2', n_jobs=-1
)

print(f"\nNested CV Complete!")
print(f"Average Nested CV R²: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}")

### Interpreting the Nested CV Results

The nested CV $R^2$ (**0.9177 ± 0.0195**) is slightly lower than our single train/test split $R^2$ (0.9418). This is expected. Nested CV gives a more conservative, honest estimate because it accounts for the uncertainty in hyperparameter selection.

**Variance:** the standard deviation of ±0.0195 tells us the model's performance is fairly stable across different data splits. A larger spread would suggest the model is more sensitive to which specific houses end up in the training set. This is the same kind of wiggle we measured with repeated splits in 17_1_6.

**Cost:** nested CV required 5 outer folds × 3 inner folds × 8 hyperparameter combinations = 120 model fits. For a dataset this size, that took just a few seconds. On larger datasets or with larger grids, the computational cost becomes significant.

## Model Comparison Summary

Let's bring everything together. Here is how every model we built in this notebook performed:

| Model | Approach | Best R² | Notes |
|---|---|---|---|
| Decision Tree (depth 5) | Single tree | 0.8220 | Interpretable but overfits easily |
| Random Forest (tuned) | Parallel ensemble | 0.8797 | Stable, robust, hard to mess up |
| HistGradientBoosting | Sequential ensemble | 0.9449 | More powerful, needs careful tuning |
| XGBoost (tuned) | Sequential + regularization | 0.9418 | Within a whisker of HGB |
| XGBoost (Nested CV) | Unbiased estimate | 0.9177 ± 0.0195 | Most honest performance estimate |

A few takeaways:

*   Steps up the complexity ladder tend to buy more accuracy, but at the cost of interpretability and computational efficiency.
*   The optimistic bias is real. Notice the gap between XGBoost's single-split score (0.9418) and its nested CV estimate (0.9177). That difference of about 2.4 points is the inflation from hyperparameter tuning, exactly what nested CV is designed to catch.


### Final Thought

In this notebook we moved from a single interpretable but overfitting decision tree ($R^2$ ≈ 0.82) to ensemble methods that approach $R^2$ ≈ 0.94, all with minimal data cleaning.

Think of OLS regression as one end of a spectrum: very interpretable, but very sensitive to data quality and feature engineering. XGBoost sits at the other end: not very interpretable, but remarkably robust to messy data and able to capture complex patterns automatically.

In **Module 18** we shift from regression to classification, where we predict categories instead of continuous values. Many of the ideas from here (cross-validation, hyperparameter tuning, feature importance) carry over directly, but the evaluation metrics and model choices will look different.